# 01 Data Preparation

This notebook corresponds to the **Data Preparation** phase of CRISP-DM.

Goals:
- load all raw CSV files
- harmonise historical schemas
- concatenate into one dataset
- clean datatypes and remove invalid stations
- save compact processed outputs (Parquet + optional CSV/GZip)

In [3]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Canonical schema and mapping

In [4]:
FINAL_COLUMNS = [
    'last_reported',
    'station_id',
    'num_bikes_available',
    'num_docks_available',
    'name',
    'address',
    'lat',
    'lon',
    'capacity',
]

OLD_TO_NEW = {
    'STATION ID': 'station_id',
    'TIME': 'last_reported',
    'NAME': 'name',
    'ADDRESS': 'address',
    'LATITUDE': 'lat',
    'LONGITUDE': 'lon',
    'BIKE_STANDS': 'capacity',
    'AVAILABLE_BIKES': 'num_bikes_available',
    'AVAILABLE_BIKE_STANDS': 'num_docks_available',
}

In [5]:
def load_and_standardize_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    # Legacy schema
    if 'STATION ID' in df.columns:
        df = df.rename(columns=OLD_TO_NEW)
        df = df.drop(columns=['LAST UPDATED', 'STATUS'], errors='ignore')

    # Keep only required columns
    keep = [c for c in FINAL_COLUMNS if c in df.columns]
    df = df[keep].copy()
    df = df.reindex(columns=FINAL_COLUMNS)
    return df


def load_all_csvs(data_dir: Path) -> pd.DataFrame:
    csv_files = sorted(data_dir.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found in {data_dir}')

    dfs = []
    for csv_file in csv_files:
        print('Loading', csv_file.name)
        dfs.append(load_and_standardize_csv(csv_file))

    return pd.concat(dfs, ignore_index=True)

## Load and clean

In [6]:
df = load_all_csvs(RAW_DIR)

df['last_reported'] = pd.to_datetime(df['last_reported'], errors='coerce')
df = df.dropna(subset=['last_reported', 'station_id'])

# Remove suspicious placeholder/test station
if 507 in set(df['station_id'].dropna().unique()):
    df = df[df['station_id'] != 507].copy()

# Remove impossible capacities
if 'capacity' in df.columns:
    df = df[df['capacity'].fillna(0) > 1].copy()

print(df.shape)
df.head()

FileNotFoundError: No CSV files found in D:\DCU\Sem2\Data analysis and mining\Project\2026-csc1144-dublin-bikes\data\raw

## Optimise dtypes before saving

In [ ]:
# integer-like fields
for col in ['station_id', 'capacity', 'num_bikes_available', 'num_docks_available']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# drop rows that are unusable after numeric conversion
df = df.dropna(subset=['station_id', 'capacity', 'num_bikes_available', 'num_docks_available']).copy()

df['station_id'] = df['station_id'].astype('int16')
df['capacity'] = df['capacity'].astype('int16')
df['num_bikes_available'] = df['num_bikes_available'].astype('int16')
df['num_docks_available'] = df['num_docks_available'].astype('int16')

df['lat'] = pd.to_numeric(df['lat'], errors='coerce').astype('float32')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce').astype('float32')

for col in ['name', 'address']:
    if col in df.columns:
        df[col] = df[col].astype('category')

df.info()

<class 'pandas.DataFrame'>
Index: 13513883 entries, 0 to 13516757
Data columns (total 9 columns):
 #   Column               Dtype         
---  ------               -----         
 0   last_reported        datetime64[us]
 1   station_id           int16         
 2   num_bikes_available  int16         
 3   num_docks_available  int16         
 4   name                 category      
 5   address              category      
 6   lat                  float32       
 7   lon                  float32       
 8   capacity             int16         
dtypes: category(2), datetime64[us](1), float32(2), int16(4)
memory usage: 438.2 MB


## Save processed dataset

In [ ]:
all_parquet = PROCESSED_DIR / 'dublin_bikes_all.parquet'
all_csv_gz = PROCESSED_DIR / 'dublin_bikes_all.csv.gz'

df.to_parquet(all_parquet, index=False)
df.to_csv(all_csv_gz, index=False, compression='gzip')

print('Saved:', all_parquet)
print('Saved:', all_csv_gz)

Saved: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/dublin_bikes_all.parquet
Saved: /home/tbeliere/Projects/DCU/2026-csc1144-dublin-bikes/data/processed/dublin_bikes_all.csv.gz


## Notes

Why Parquet + dtype optimisation?
- Parquet is columnar and compressed, so it is much smaller and much faster to load than CSV.
- Downcasting integers/floats and converting repeated strings to `category` can reduce memory and disk usage substantially.